# Introduction:

### Problem:
I listen to less popular musical artists and I like showing them to my friends. However my music taste varies a lot and the artists I listen to touch on a large range of topics, including some sensitive topics. My roommate recently celebrated her anniversary and so I wanted to see what romantic songs I listen to that I could recommend to her.

This inspired the idea of a lyric-topic based song recommender using Natural Langauge Processing. This tool can identify topics in song lyrics and from there, be able to search for new songs by inputing a keyword and returning songs about the most related topic. When applied to my own dataset, I can find songs to recommend to others, and when applied to a general dataset I can find new songs for me to listen to.  

Sometimes when listening to music it can be easy to gauge topic of a song just by listening to it, but this isn't always the case. Some songs can be too fast or too melodic to catch the lyrics, while others the vocals can be drowned out by other sounds due to stylistic choices or poor audio quality. In all of these cases, you need to look at the lyrics themselves to really get an idea of what the topic of the song is.  
However, it can take a long time to read through song lyrics just to find new music to listen to on a topic that you want. While one could take a more simplistic approach and just parse through the song lyrics and identify songs with the same word used, this approach has a major fallback. Not all worship songs may not necessarily use the word "worship". Or similarly, if someone looked for songs about drinking, different songs may use different words as they may reference different drinks.  
One way to remedy this is to create lists of related words, but that would require you to know all of the related words and groupings that would appear in the dataset before you search it, which isn't just isn't feasible.


---

### How we plan on solving it:
Lyric Topic Identifier and Recommender Using NLP and LDA
  - We plan on using a variety of tools, including the The Natural Language Toolkit (NLTK), Scikit Learn's term frequency–inverse document frequency (TF-IDF) tool, and Scikit Learn's Latent Dirichlet Allocation (LDA) tool.
    - NLTK is a comprehensive Python library used for working with human language data, or text. Using this tool kit we can create an robust recommendation system based on the lyrics and other variables of a song to reccomend music that covers certain user identified topics.
    - Scikit Learn's TF-IDF tools are used to create our weighted matrix of each word's frequency
    - LDA uses probability to create a set of topics based on the distribution of words

We took this approach specifically because LDA takes into account that topics are often not fully independent, which is especially true for song lyrics. We go further into our choices to take the approaches we do as they come up later.


### Our Data
We first built our program using a dataset from Kaggle, and then further tested it on a hand-built dataset of songs that I personally enjoy. The Kaggle dataset is great for finding new music because it is a very large dataset with a wide range of songs. To solve our actual problem though, we needed to use the dataset of what music I listen to so I can find topics and make recommendations within my own music. We didn't work with this dataset at the start, because it is a much more limited dataset, and we wanted to make sure the program worked regardless.

In [1]:
#import packages
import pandas as pd

#used in BoW
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
from sklearn.feature_extraction import text
from nltk.stem import WordNetLemmatizer

#used in LDA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

#used in recommender
from sklearn.metrics.pairwise import cosine_similarity

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Our Dataset
Our training dataset is the Spotify Million Song Dataset from [Kaggle](https://www.kaggle.com/datasets/notshrirang/spotify-million-song-dataset/data).
This dataset has a large amount of songs and lyrics which makes this a good dataset to start off of.
- The key columns in the dataset include:
```
    song: The title of the song.
    artists: The artist(s) who performed the song.
    text: Likely the song's lyrics or some textual description of the track.
```

In [ ]:
#import whole dataset
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('spotify_millsongdata.csv')

#taking a quick look at the data
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [220]:
#removing the link column since it isn't helpful
df.drop(columns=['link'], inplace=True)

df.tail()

,artist,song,text
57645,Ziggy Marley,Good Old Days,Irie days come on play \r\nLet the angels fly...
57646,Ziggy Marley,Hand To Mouth,Power to the workers \r\nMore power \r\nPowe...
57647,Zwan,Come With Me,all you need \r\nis something i'll believe \...
57648,Zwan,Desire,northern star \r\nam i frightened \r\nwhere ...
57649,Zwan,Heartsong,come in \r\nmake yourself at home \r\ni'm a ...


In [221]:
df.shape

(57650, 3)

In [222]:
#check for misssing values - none missing so none need dropped
df.isnull().sum()

,0
artist,0
song,0
text,0


In [223]:
# Taking a random sample of 5000 rows and reset index for easier processing
# set random state for reproducability
df = df.sample(5000, random_state=100).reset_index(drop=True)

### Text Cleaning and Pre-Processing
When going through the text column specifically, we want to remove any characters that aren't part of words. This includes newlines, punctuation, and numbers. Some songs also have song section identifiers within brackets, such as "[Verse 1]" and so we want to remove any text inside these brackets when we remove them.  
Another important step we took here was removing any 1 or 2 letter words. The idea of the "three-letter-rule" is an observation that in English, almost all words shorter than 3 letters long are function words, such as pronouns like "I", prepositions like "by", or neutral verbs like "do". Some exceptions to this that we may find are vocalizations like "la" or "oh", or parts of contractions like "ll" or "t", since we are also removing apostrophes. All of these are words that we don't want included in our model, and by removing them now it will make adjusting our stopwords list much easier.

In [224]:
def df_text_trim(df):
  df['text'] = df['text'].str.lower().replace("(\\n)|\'|’|(\\r)", ' ', regex=True).replace(r'(\[.*?\])|\W*\b\w{1,2}\b','',regex=True).replace('[^a-zA-Z ]','',regex=True)

In [225]:
# cleaning up the text column
# removed all text within brackets, newlines, special charcters, numbers
# also removed words under 3 letters long because most to all are sounds or function words, both of which we'd want removed
df_text_trim(df)
df.head()

,artist,song,text
0,Elton John,Dear God,dear god are you there can you hear you car...
1,Religious Music,All Creatures Of Our God And King,all creatures our god and king lift your vo...
2,Pogues,Down In The Ground Where The Dead Men Go,hello boys been away bit holiday the land wher...
3,Dewa 19,Cukup Siti Nurbaya,masih ada belenggu ruang cinta meresap kin...
4,Linkin Park,Castle Of Glass,take down the river bend take down the figh...


In [227]:
df['text'][9]

' could while away the hours    conferrin  with the flowers    consultin  with the rain    and head scratchin     while thoughts were busy hatchin only had brain unravel any riddle    for any individd trouble pain    with the thoughts thinkin could another lincoln only had brain could tell you why    the ocean near the shore could think things never thunk before    and then sit and think some more would not just nothin head all full stuffin heart all full pain    perhaps deserve you    and even worthy serve you only had only had brain    '

##Bag Of Words Approach
We used a bag of words approach to highlight key words that are most frequently repeated in each song. We used this to locate frequently occurring words that are filler, names, vocalizations, or non-key, such as prepositions or neutral verbs, that we didn't already remove in our process to clean the text. From what we see in the bag of words approach, we can find what words to add to our stopwords list.

When tokeninzing our data in this step, we also applied a lemmatizer so that words with the same root are treated as the same word. Using a lemmatizer rather than a stemmer is useful because it keeps the words as actual words. By re-joining our tokens and saving it in place of our original text, we have a processed version of the text to build the LDA off of later so it's working with cleaner text.

In [228]:
#some of the additional stopwords are other spellings of words already in the stopwords list since the spelling isn't always perfect
english_stopwords = nltk.corpus.stopwords.words('english')
my_stop_words = (['ooh', 'doo', 'wop','aye','just', "till", "cant", "got", 'dont', 'bala', 'shes', 'yaga','aaah', 'aah', 'ahh','noah', 'maria', 'diana', 'dan','daniel','arin','avidan','hanson','bum','bah','let','harry','del','ann','johnny','ding','ing','woah','non','bop','che','nuh','one','two','three','four'])
english_stopwords.extend(my_stop_words)
lemmatizer = WordNetLemmatizer()

In [229]:
#word tokenizer was used because it splits similar to how tf-idf splits words
def tokenize(text):
  token = word_tokenize(text.lower())
  a = [lemmatizer.lemmatize(word) for word in token if word not in english_stopwords]
  return a

def bag_of_words(text):
  a = tokenize(text)
  bag = Counter(a)
  terms = bag.most_common(10)
  return terms

def processed_string(text):
  a = tokenize(text)
  b = ' '.join(a)
  return b

In [230]:
# use the tokenize function on the text column of our dataset
df['text']= df['text'].apply(lambda x: processed_string(x))
df['terms']= df['text'].apply(lambda x: bag_of_words(x))

In [231]:
df.head(10)

,artist,song,text,terms
0,Elton John,Dear God,dear god hear care dear god le perfect far fre...,"[(dear, 12), (god, 12), (way, 5), (take, 4), (..."
1,Religious Music,All Creatures Of Our God And King,creature god king lift voice sing alleluia all...,"[(alleluia, 29), (praise, 25), (thou, 8), (ref..."
2,Pogues,Down In The Ground Where The Dead Men Go,hello boy away bit holiday land river freely f...,"[(ground, 4), (fell, 3), (ran, 3), (hell, 3), ..."
3,Dewa 19,Cukup Siti Nurbaya,masih ada belenggu ruang cinta meresap kini di...,"[(yang, 6), (bukan, 4), (ada, 3), (cinta, 3), ..."
4,Linkin Park,Castle Of Glass,take river bend take fighting end wash poison ...,"[(see, 9), (cause, 4), (crack, 4), (castle, 4)..."
5,Donna Summer,I Don't Wanna Get Hurt,need friend tell thing already know like best ...,"[(want, 16), (get, 15), (hurt, 15), (know, 8),..."
6,Ying Yang Twins,By Myself,ying yang twin ghetto public service announcme...,"[(get, 10), (nigga, 7), (fuck, 7), (ohh, 5), (..."
7,Olivia Newton-John,It's Christmas Time Down Under,see sun sky always riding high christmas time ...,"[(time, 8), (christmas, 7), (see, 4), (sky, 3)..."
8,Judy Garland,You Go To My Head,head linger like haunting refrain find spinnin...,"[(like, 5), (head, 4), (find, 2), (thought, 2)..."
9,"Harry Connick, Jr.",If I Only Had A Brain,could away hour conferrin flower consultin rai...,"[(could, 4), (brain, 3), (head, 2), (thought, ..."


##Latent Dirichlet Association
Now that we have established an expanded stopwords list, we can start actually looking for topics within our song lyrics. To do this, we used Latent Dirichlet Association (LDA). LDA does not assume an entry, in this case a single song's lyrics to be ablout a single topic. Instead it may sort a song to be 80% about one topic, 15% about another, and 5% about a third. This helps it provide better topic identification for our use over purely grouping together the keywords we found earlier when doing our Bag of Words Approach.   

LDA's approach marking each item's distribution amongst topics is especially helpful as we apply it to song lyrics, as songs often mix topics. For example, we may find some songs about dancing and love, while others are about dancing and partying.



To do the LDA, first we needed to make a term frequency–inverse document frequency (TF-IDF) matrix since it will give us a more refined, and weighted result. When creating it, we specified to remove the stopwords that we identified earlier, as well as words that appeared in too many or too few of songs.

In [232]:
stop_words = stopwords.words('english')
stop_words.extend(my_stop_words)

In [233]:
def vector_for_lda(df):
  tfidf = TfidfVectorizer(stop_words=stop_words, min_df = 3, max_df= .75)
  doc_term_matrix = tfidf.fit_transform(df['text'])
  print(f'Rows: {doc_term_matrix.shape[0]}, Columns: {doc_term_matrix.shape[1]}')
  return doc_term_matrix, tfidf

In [234]:
doc_term_matrix, tfidf = vector_for_lda(df)

Rows: 5000, Columns: 7157


After the matrix is created we choose how many topics we want all of the songs sorted into. One of the tricky things about working with LDA is that LDA will not identify the number of topics for you. Instead you have to input how many topics you want it to group your data into. To get a good number of topics that aren't too broad or narrow we started with 6 and through trial and error tested to see how our outcomes were with different numbers.

We then run the LDA  on our matrix of terms. From there, we create a dataframe of the weight of topics for each song.

In [235]:
def lda_topics(doc_term_matrix, num_topics):
  lda = LatentDirichletAllocation(n_components=num_topics, random_state=0)
  matrix = lda.fit_transform(doc_term_matrix)

  col_names = [f'Topic {x}' for x in range(1, num_topics+1)]
  doc_topic_df = pd.DataFrame(matrix, columns=col_names)
  return lda, doc_topic_df, matrix

In [274]:
lda, doc_topic_df, matrix = lda_topics(doc_term_matrix, 20)
doc_topic_df.head()

,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,Topic 8,Topic 9,Topic 10,Topic 11,Topic 12,Topic 13,Topic 14,Topic 15,Topic 16,Topic 17,Topic 18,Topic 19,Topic 20
0,0.009987,0.073459,0.009987,0.009987,0.009987,0.009987,0.009987,0.042684,0.009987,0.009987,0.009987,0.009987,0.009987,0.009987,0.009987,0.009987,0.714071,0.009987,0.009987,0.009987
1,0.011952,0.011952,0.011952,0.011952,0.011952,0.011952,0.011952,0.011952,0.018593,0.011952,0.011952,0.239298,0.011952,0.011952,0.011952,0.011952,0.314996,0.098083,0.149751,0.011952
2,0.005464,0.040256,0.005464,0.005464,0.005464,0.005464,0.005464,0.019895,0.084531,0.005464,0.005464,0.005464,0.005464,0.005464,0.005464,0.005464,0.767897,0.005464,0.005464,0.005464
3,0.013878,0.013878,0.548092,0.013878,0.013878,0.013878,0.013878,0.013878,0.013878,0.013878,0.118620,0.049408,0.013878,0.013878,0.050372,0.013878,0.025338,0.013878,0.013878,0.013878
4,0.008539,0.032124,0.008539,0.008539,0.008539,0.008539,0.008539,0.031805,0.008539,0.008539,0.008539,0.008539,0.008539,0.008539,0.008539,0.008539,0.765155,0.034290,0.008539,0.008539


To actually get an idea of what these topics are, we output the top 10 songs for each topic.

In [275]:
def get_top_words(lda, tfidf, num_words):
  for topic, words in enumerate(lda.components_):
    word_total = words.sum()
    sorted_words = words.argsort()[::-1]
    print(f'Topic {topic+1}:')
    for i in range(0, num_words):
      word = tfidf.get_feature_names_out()[sorted_words[i]]
      word_weight = words[sorted_words[i]]
      print(f'{word} ({word_weight:.2f})', end=' ')
    print('')

In our output below we can see the topics and some of their keywords. There may still be some non helpful words since we may not have all the ones we need in our list. There are also a few songs that aren't in English which our model is not expecting.

In [276]:
get_top_words(lda, tfidf,5)

Topic 1:
healing (2.48) raining (2.30) physical (2.01) lazy (1.83) naughty (1.45) 
Topic 2:
lonesome (8.01) soldier (5.77) emotion (5.14) magic (4.69) fallin (4.25) 
Topic 3:
gloria (2.80) yang (2.56) kau (2.41) orleans (2.38) candy (1.73) 
Topic 4:
loneliness (4.64) hallelujah (4.13) anytime (3.99) expect (3.98) paradise (3.36) 
Topic 5:
hai (2.60) business (2.48) louise (2.21) mein (1.89) whiskey (1.83) 
Topic 6:
sighing (1.87) excited (1.85) serf (1.71) grieving (1.67) rudolph (1.60) 
Topic 7:
holiday (3.60) pum (3.46) blossom (2.72) frankie (2.12) poetry (1.88) 
Topic 8:
nigga (21.85) bitch (14.13) shit (13.97) fuck (12.97) drop (7.01) 
Topic 9:
ave (3.06) dee (2.01) cheatin (1.48) gleam (1.38) summertime (1.32) 
Topic 10:
sooner (3.06) noel (2.23) painful (2.13) youre (1.89) bluebird (1.81) 
Topic 11:
papa (4.15) breakin (2.73) ordinary (2.69) drifting (2.39) fence (2.37) 
Topic 12:
teacher (3.16) cadillac (2.03) memphis (2.02) yer (1.87) salt (1.79) 
Topic 13:
kentucky (2.36) voo

In [277]:
#Adding a column of the highest matched topic for each
df['topic'] = doc_topic_df.idxmax(axis=1).replace('Topic ', '', regex=True)
df.head()

,artist,song,text,terms,topic
0,Elton John,Dear God,dear god hear care dear god le perfect far fre...,"[(dear, 12), (god, 12), (way, 5), (take, 4), (...",17
1,Religious Music,All Creatures Of Our God And King,creature god king lift voice sing alleluia all...,"[(alleluia, 29), (praise, 25), (thou, 8), (ref...",17
2,Pogues,Down In The Ground Where The Dead Men Go,hello boy away bit holiday land river freely f...,"[(ground, 4), (fell, 3), (ran, 3), (hell, 3), ...",17
3,Dewa 19,Cukup Siti Nurbaya,masih ada belenggu ruang cinta meresap kini di...,"[(yang, 6), (bukan, 4), (ada, 3), (cinta, 3), ...",3
4,Linkin Park,Castle Of Glass,take river bend take fighting end wash poison ...,"[(see, 9), (cause, 4), (crack, 4), (castle, 4)...",17


In [278]:
#Creating a dataframe of each word's frequency within each topic
topic_words= pd.DataFrame(lda.components_, columns=tfidf.get_feature_names_out()).transpose()
topic_words.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
abandon,0.05,0.050000,0.05,0.827635,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.050000,0.05,0.05,0.05,0.05,0.050000,0.05,0.05,0.05
abandoned,0.05,0.050000,0.05,0.050000,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.050000,0.05,0.05,0.05,0.05,2.488972,0.05,0.05,0.05
abba,0.05,0.050000,0.05,0.050000,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.050000,0.05,0.05,0.05,0.05,0.826952,0.05,0.05,0.05
abc,0.05,0.296592,0.05,0.050000,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.122029,0.05,0.05,0.05,0.05,0.050001,0.05,0.05,0.05
ability,0.05,0.050000,0.05,0.050000,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.050000,0.05,0.05,0.05,0.05,0.419442,0.05,0.05,0.05


Here we did a little check to see how the word's most common topics were distributed, just to see. We don't expect a normal distribution since there likely isn't a normal distribution of topics amongst the songs.

In [279]:
top_weight = topic_words.idxmax(axis=1)
top_weight.value_counts()

,count
16,2901
7,896
1,476
3,449
10,267
18,235
11,214
19,190
17,173
6,162


In [265]:
def song_topic_recommender(term, num=5):
  try:
    term_dist = topic_words.loc[term]
    topic = str(term_dist.idxmax()+1)
    return df[df['topic'] == topic][0:num].sample(num)

  except:
    return 'Topic not found'

In [281]:
song_topic_recommender('god')

,artist,song,text,terms,topic
2,Pogues,Down In The Ground Where The Dead Men Go,hello boy away bit holiday land river freely f...,"[(ground, 4), (fell, 3), (ran, 3), (hell, 3), ...",17
5,Donna Summer,I Don't Wanna Get Hurt,need friend tell thing already know like best ...,"[(want, 16), (get, 15), (hurt, 15), (know, 8),...",17
0,Elton John,Dear God,dear god hear care dear god le perfect far fre...,"[(dear, 12), (god, 12), (way, 5), (take, 4), (...",17
1,Religious Music,All Creatures Of Our God And King,creature god king lift voice sing alleluia all...,"[(alleluia, 29), (praise, 25), (thou, 8), (ref...",17
4,Linkin Park,Castle Of Glass,take river bend take fighting end wash poison ...,"[(see, 9), (cause, 4), (crack, 4), (castle, 4)...",17


# Applying Our Code to My Music
Now that we finally have a working program, we can apply it to the dataset actually relevant to our problem. I put together this dataset from songs some of my favorite songs, as well as songs I remember listening to recently,songs that Spotify listed on my "On Repeat" Playlist, and songs that I've recently listened to, removing any duplicates, such as any remastered versions.

In [ ]:
df = pd.read_csv('songs.csv', encoding='cp1252')
df.drop(columns=['Unnamed: 4'], inplace=True)
df.rename(columns={'title': 'song'}, inplace=True)
df.head()

,artist,album,song,text
0,Will Wood and the Tapeworms,Everything is a Lot,6up 5oh Cop-Out (Pro/Con),"[Verse 1]\nSix-up, five-oh, pigs come, I cop n..."
1,Will Wood and the Tapeworms,Everything is a Lot,"Skeleton Appreciation Day in Vestal, NY (Bones)",[Verse 1]\nTo cut down on my silhouette\nMy fa...
2,Will Wood and the Tapeworms,Everything is a Lot,Front Street,"[Intro]\n""Quick while she turns her back, slip..."
3,Will Wood and the Tapeworms,Everything is a Lot,¡Aikido! (Neurotic/Erotic),[Verse 1]\nI apologize for playing with your e...
4,Will Wood and the Tapeworms,Everything is a Lot,White Knuckle Jerk (Where Do You Get Off?),[Verse 1]\nShe’s got the eyes of a snake\nLoad...


In [134]:
#cleaning our data
df_text_trim(df)
df.head()

,artist,album,song,text
0,Will Wood and the Tapeworms,Everything is a Lot,6up 5oh Cop-Out (Pro/Con),six five pigs come cop the blotter shows they...
1,Will Wood and the Tapeworms,Everything is a Lot,"Skeleton Appreciation Day in Vestal, NY (Bones)",cut down silhouette favorite foods are smoke ...
2,Will Wood and the Tapeworms,Everything is a Lot,Front Street,quick while she turns her back slip meatpac...
3,Will Wood and the Tapeworms,Everything is a Lot,¡Aikido! (Neurotic/Erotic),apologize for playing with your eyes but obse...
4,Will Wood and the Tapeworms,Everything is a Lot,White Knuckle Jerk (Where Do You Get Off?),she got the eyes snake loaded dice raising st...


We chose to split this dataset into 5 topics because

In [136]:
#tokenizing and seeing our topic groups
df['text']= df['text'].apply(lambda x: processed_string(x))
df['terms']= df['text'].apply(lambda x: bag_of_words(x))
doc_term_matrix, tfidf = vector_for_lda(df)
lda, doc_topic_df, matrix = lda_topics(doc_term_matrix, 5)
get_top_words(lda, tfidf, 10)

Rows: 120, Columns: 1442
Topic 1:
bone (3.59) replace (2.14) well (2.09) na (1.91) come (1.88) matter (1.78) take (1.75) wan (1.57) please (1.54) red (1.53) 
Topic 2:
remember (2.49) obsessed (1.98) neurotic (1.40) religion (1.31) erotic (1.30) destroy (1.22) enjoy (1.17) forgot (1.09) profound (0.93) fuck (0.85) 
Topic 3:
know (6.66) could (4.75) like (4.70) never (4.52) one (4.34) get (4.27) enough (3.64) day (3.37) come (3.30) die (3.19) 
Topic 4:
bad (2.18) norm (1.93) eye (1.85) chemical (1.14) singing (1.12) particle (0.89) shoe (0.77) mile (0.76) whatever (0.74) distraction (0.70) 
Topic 5:
wish (2.19) sunshine (1.67) moon (1.25) big (1.24) girlfriend (1.10) boyfriend (1.10) could (1.04) night (1.02) girl (0.97) fucking (0.96) 


In [138]:
df['topic'] = doc_topic_df.idxmax(axis=1).replace('Topic ', '', regex=True)
topic_words= pd.DataFrame(lda.components_, columns=tfidf.get_feature_names_out()).transpose()
song_topic_recommender('love',5)

,artist,album,song,text,topic,terms
8,Will Wood and the Tapeworms,Everything is a Lot,Lysergide Daydream,wan na picture postcard pouring pitcher backya...,3,"[(place, 9), (wan, 6), (na, 6), (away, 5), (ca..."
4,Will Wood and the Tapeworms,Everything is a Lot,White Knuckle Jerk (Where Do You Get Off?),eye snake loaded dice raising stake cash cow b...,3,"[(never, 8), (know, 8), (like, 6), (get, 6), (..."
2,Will Wood and the Tapeworms,Everything is a Lot,Front Street,quick turn back slip meatpack plant gutterside...,3,"[(drink, 7), (front, 6), (street, 6), (let, 5)..."
5,Will Wood and the Tapeworms,Everything is a Lot,(Cover This Song) A Little Bit Mine,say still woman turned someone never could lov...,3,"[(little, 19), (bit, 14), (back, 6), (mine, 6)..."
7,Will Wood and the Tapeworms,Everything is a Lot,RED MOON,red red moon keep rising sunset soon indeed bl...,3,"[(red, 18), (might, 10), (moon, 9), (around, 9..."


#Findings
The model we created says I should suggests that my friend listen to the songs "Lysergide Daydream", "White Knuckle Jerk", "Front Street", "A Little Bit Mine", and "RED MOON".
Red Moon and Front Street aren't the best suggestions given the keyword "love", but the other three are great choices.
This shows the model does work but it still has some weaknesses.   
In future versions, I'd try to expand the training data so the model isn't swayed by a particular artist's songwriting style. In this future version, it might be better to incorporate a Support Vector Machine to classify the songs into their primary topics to get more accurate groupings.